# Module 14: No-Code Agents with n8n

**Companion notebook for**: `14-n8n-no-code.html`

## Learning Objectives
- Understand n8n architecture and how it fits into the GenAI ecosystem
- Install and interact with n8n programmatically via its REST API
- Build AI workflows: webhook triggers, LLM nodes, and structured outputs
- Compare no-code (n8n) vs code (Python) approaches for the same task
- Design common workflow patterns: chatbots, document processing, data pipelines
- Manage credentials, error handling, and production deployment

## Prerequisites
- Docker installed (for n8n local setup)
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Basic understanding of REST APIs and webhooks

---
## Setup & Dependencies

In [ ]:
%pip install -q openai httpx requests pydantic

In [ ]:
import os
import json
import time
import textwrap
from typing import Any

import httpx
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

# Verify API key is available
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

# n8n configuration -- update if your instance runs elsewhere
N8N_BASE_URL = os.environ.get("N8N_BASE_URL", "http://localhost:5678")
N8N_API_KEY = os.environ.get("N8N_API_KEY", "")  # Set after enabling API access in n8n

print("All imports successful.")
print(f"n8n target: {N8N_BASE_URL}")
print(f"n8n API key configured: {bool(N8N_API_KEY)}")

---
## Section 1: What Is n8n & Installation

n8n (pronounced "n-eight-n" or "nodemation") is an open-source workflow automation platform.
Three features set it apart from Zapier or Make:

| Feature | Why It Matters |
|---|---|
| **Self-hostable** | Data never leaves your infrastructure (HIPAA, SOC 2, GDPR) |
| **Code node** | Write arbitrary JS/Python when the visual UI is not enough |
| **Native AI nodes** | Dedicated nodes for LLM chat, embeddings, vector stores, agents |

The core concept is a **workflow** -- a visual graph of connected nodes. Each node does one action:
receive a webhook, call an API, transform data, send an email, query a database, or invoke an LLM.
Data flows through connections from node to node.

In [ ]:
# ============================================================
# Option 1: Install n8n with Docker (recommended)
# Run this in a terminal, NOT in Jupyter.
# ============================================================

docker_run_cmd = """
docker run -it --rm \\
  --name n8n \\
  -p 5678:5678 \\
  -v n8n_data:/home/node/.n8n \\
  -e GENERIC_TIMEZONE="America/Chicago" \\
  n8nio/n8n
""".strip()

print("Run this command in your terminal to start n8n:")
print("=" * 55)
print(docker_run_cmd)
print("=" * 55)
print()
print("Then open http://localhost:5678 in your browser.")
print()
print("Alternative -- install via npm:")
print("  npm install n8n -g")
print("  n8n start")

In [ ]:
# ============================================================
# Production Docker Compose (save as docker-compose.yml)
# Uses PostgreSQL + Redis for queue-mode execution
# ============================================================

docker_compose_yaml = """
version: "3.8"
services:
  n8n:
    image: n8nio/n8n:latest
    restart: always
    ports:
      - "5678:5678"
    environment:
      - DB_TYPE=postgresdb
      - DB_POSTGRESDB_HOST=postgres
      - DB_POSTGRESDB_DATABASE=n8n
      - DB_POSTGRESDB_USER=n8n
      - DB_POSTGRESDB_PASSWORD=${DB_PASSWORD}
      - EXECUTIONS_MODE=queue
      - QUEUE_BULL_REDIS_HOST=redis
      - N8N_ENCRYPTION_KEY=${ENCRYPTION_KEY}
      - WEBHOOK_URL=https://n8n.yourdomain.com/
      - N8N_BASIC_AUTH_ACTIVE=true
      - N8N_BASIC_AUTH_USER=${N8N_USER}
      - N8N_BASIC_AUTH_PASSWORD=${N8N_PASSWORD}
      - EXECUTIONS_DATA_MAX_AGE=720
      - GENERIC_TIMEZONE=America/Chicago
    volumes:
      - n8n_data:/home/node/.n8n
    depends_on:
      - postgres
      - redis

  postgres:
    image: postgres:16
    restart: always
    environment:
      POSTGRES_DB: n8n
      POSTGRES_USER: n8n
      POSTGRES_PASSWORD: ${DB_PASSWORD}
    volumes:
      - postgres_data:/var/lib/postgresql/data

  redis:
    image: redis:7-alpine
    restart: always

  n8n-worker:
    image: n8nio/n8n:latest
    command: worker
    restart: always
    environment:
      - DB_TYPE=postgresdb
      - DB_POSTGRESDB_HOST=postgres
      - DB_POSTGRESDB_DATABASE=n8n
      - DB_POSTGRESDB_USER=n8n
      - DB_POSTGRESDB_PASSWORD=${DB_PASSWORD}
      - EXECUTIONS_MODE=queue
      - QUEUE_BULL_REDIS_HOST=redis
      - N8N_ENCRYPTION_KEY=${ENCRYPTION_KEY}
    depends_on:
      - postgres
      - redis

volumes:
  n8n_data:
  postgres_data:
""".strip()

print("Production docker-compose.yml:")
print(docker_compose_yaml)

---
## Section 2: Workflow JSON Structure

n8n workflows are stored as JSON and can be version-controlled. A workflow consists of:
- **nodes** -- each node has a type, parameters, and a position on the canvas
- **connections** -- define how data flows between nodes
- **settings** -- execution timeout, error workflow, retry behavior

Below we define a minimal AI workflow in pure JSON: **Webhook --> OpenAI Chat --> Respond**.

In [ ]:
# ============================================================
# Minimal n8n AI Workflow JSON
# Webhook (POST /chat) -> OpenAI Chat -> Respond to Webhook
# ============================================================

ai_chat_workflow = {
    "name": "AI Chat Endpoint",
    "nodes": [
        {
            "id": "webhook-1",
            "name": "Webhook",
            "type": "n8n-nodes-base.webhook",
            "typeVersion": 1.1,
            "position": [250, 300],
            "parameters": {
                "httpMethod": "POST",
                "path": "chat",
                "responseMode": "responseNode"
            },
            "webhookId": "chat-endpoint"
        },
        {
            "id": "openai-1",
            "name": "OpenAI Chat",
            "type": "@n8n/n8n-nodes-langchain.openAi",
            "typeVersion": 1.2,
            "position": [500, 300],
            "parameters": {
                "model": "gpt-4o",
                "prompt": "={{ $json.body.message }}",
                "systemMessage": "You are a helpful assistant.",
                "options": {
                    "temperature": 0.7,
                    "maxTokens": 1024
                }
            },
            "credentials": {
                "openAiApi": {"id": "1", "name": "OpenAI account"}
            }
        },
        {
            "id": "respond-1",
            "name": "Respond to Webhook",
            "type": "n8n-nodes-base.respondToWebhook",
            "typeVersion": 1,
            "position": [750, 300],
            "parameters": {
                "respondWith": "json",
                "responseBody": "={{ { response: $json.text } }}"
            }
        }
    ],
    "connections": {
        "Webhook": {
            "main": [[{"node": "OpenAI Chat", "type": "main", "index": 0}]]
        },
        "OpenAI Chat": {
            "main": [[{"node": "Respond to Webhook", "type": "main", "index": 0}]]
        }
    },
    "settings": {
        "executionTimeout": 300,
        "saveManualExecutions": True
    }
}

print("Workflow JSON structure:")
print(json.dumps(ai_chat_workflow, indent=2))

In [ ]:
# ============================================================
# Inspect the workflow structure programmatically
# ============================================================

def describe_workflow(workflow: dict) -> None:
    """Pretty-print a summary of an n8n workflow."""
    print(f"Workflow: {workflow['name']}")
    print(f"Nodes ({len(workflow['nodes'])}):")
    for node in workflow["nodes"]:
        print(f"  [{node['id']}] {node['name']}")
        print(f"         type: {node['type']}")
        params = node.get("parameters", {})
        for k, v in list(params.items())[:3]:
            print(f"         {k}: {v}")
    print(f"\nConnections:")
    for src, targets in workflow["connections"].items():
        for chain in targets.get("main", []):
            for conn in chain:
                print(f"  {src} --> {conn['node']}")

describe_workflow(ai_chat_workflow)

---
## Section 3: n8n REST API Interaction

n8n exposes a REST API that lets you manage workflows programmatically.
Enable it in n8n Settings > API, then generate an API key.

Key endpoints:
| Method | Endpoint | Purpose |
|---|---|---|
| `GET` | `/api/v1/workflows` | List all workflows |
| `POST` | `/api/v1/workflows` | Create a new workflow |
| `POST` | `/api/v1/workflows/{id}/activate` | Activate a workflow |
| `POST` | `/api/v1/executions` | Trigger a workflow execution |
| `GET` | `/api/v1/executions/{id}` | Get execution status/result |

In [ ]:
# ============================================================
# n8n API Client
# ============================================================

class N8NClient:
    """Lightweight client for the n8n REST API."""

    def __init__(self, base_url: str, api_key: str):
        self.base_url = base_url.rstrip("/")
        self.headers = {
            "X-N8N-API-KEY": api_key,
            "Content-Type": "application/json",
        }

    # -- workflows ------------------------------------------------

    def list_workflows(self) -> list[dict]:
        """Return all workflows."""
        resp = requests.get(
            f"{self.base_url}/api/v1/workflows",
            headers=self.headers, timeout=15,
        )
        resp.raise_for_status()
        return resp.json().get("data", [])

    def create_workflow(self, workflow_json: dict) -> dict:
        """Create a workflow from JSON."""
        resp = requests.post(
            f"{self.base_url}/api/v1/workflows",
            headers=self.headers,
            json=workflow_json,
            timeout=15,
        )
        resp.raise_for_status()
        return resp.json()

    def activate_workflow(self, workflow_id: str) -> dict:
        """Activate a workflow so its triggers are live."""
        resp = requests.post(
            f"{self.base_url}/api/v1/workflows/{workflow_id}/activate",
            headers=self.headers, timeout=15,
        )
        resp.raise_for_status()
        return resp.json()

    def deactivate_workflow(self, workflow_id: str) -> dict:
        """Deactivate a workflow."""
        resp = requests.post(
            f"{self.base_url}/api/v1/workflows/{workflow_id}/deactivate",
            headers=self.headers, timeout=15,
        )
        resp.raise_for_status()
        return resp.json()

    # -- executions -----------------------------------------------

    def get_executions(self, workflow_id: str = "", limit: int = 10) -> list[dict]:
        """List recent executions, optionally filtered by workflow."""
        params: dict[str, Any] = {"limit": limit}
        if workflow_id:
            params["workflowId"] = workflow_id
        resp = requests.get(
            f"{self.base_url}/api/v1/executions",
            headers=self.headers, params=params, timeout=15,
        )
        resp.raise_for_status()
        return resp.json().get("data", [])

    # -- webhook trigger ------------------------------------------

    def trigger_webhook(self, path: str, payload: dict) -> dict:
        """Send a POST request to a webhook-triggered workflow."""
        resp = requests.post(
            f"{self.base_url}/webhook/{path}",
            json=payload, timeout=30,
        )
        resp.raise_for_status()
        return resp.json()


# Instantiate the client (will only work when n8n is running)
n8n = N8NClient(N8N_BASE_URL, N8N_API_KEY)
print("N8NClient ready.")
print("Note: API calls below require a running n8n instance with API access enabled.")

In [ ]:
# ============================================================
# Demo: Create & trigger a workflow via the API
# Uncomment and run when n8n is running locally.
# ============================================================

# Step 1: Create the workflow
# created = n8n.create_workflow(ai_chat_workflow)
# workflow_id = created["id"]
# print(f"Created workflow: {workflow_id}")

# Step 2: Activate so the webhook is live
# n8n.activate_workflow(workflow_id)
# print("Workflow activated")

# Step 3: Trigger the webhook
# result = n8n.trigger_webhook("chat", {"message": "Explain RAG in one sentence."})
# print(f"Response: {result}")

# Step 4: Check execution history
# executions = n8n.get_executions(workflow_id, limit=5)
# for ex in executions:
#     print(f"  Execution {ex['id']}: status={ex['status']}")

print("[Demo code is commented out -- uncomment when n8n is running]")

---
## Section 4: AI & LLM Nodes in n8n

n8n provides dedicated AI nodes that use LangChain under the hood:

| Node | Category | Purpose |
|---|---|---|
| OpenAI Chat Model | LLM | GPT-4o, GPT-4o-mini chat completions |
| Anthropic Chat Model | LLM | Claude 3.5 Sonnet, Haiku |
| Ollama Chat Model | LLM | Local models via Ollama |
| AI Agent | Agent | ReAct agent with configurable tools |
| Vector Store (Pinecone, Qdrant) | RAG | Store and query embeddings |
| Embeddings (OpenAI, Cohere) | RAG | Generate text embeddings |
| Text Splitter | RAG | Chunk documents for indexing |
| Document Loader | RAG | Load PDF, CSV, JSON, web pages |
| Output Parser | Utility | Parse structured output (JSON, lists) |
| Memory (Buffer, Summary) | Utility | Conversation history management |
| Code (JS/Python) | Custom | Run arbitrary code in workflow |

The **AI Agent** node is the most powerful: it creates a complete ReAct agent.
You connect five things to it:
1. **Chat Model** -- the LLM brain
2. **Tools** -- actions the agent can take (any n8n node)
3. **Memory** -- conversation history
4. **Output Parser** -- optional structured output
5. **Trigger** -- how the agent receives input

In [ ]:
# ============================================================
# n8n AI Agent Workflow JSON
# Webhook -> AI Agent (with tools) -> Respond
# ============================================================

agent_workflow = {
    "name": "Customer Support Agent",
    "nodes": [
        {
            "id": "trigger-1",
            "name": "Chat Webhook",
            "type": "n8n-nodes-base.webhook",
            "typeVersion": 1.1,
            "position": [200, 300],
            "parameters": {
                "httpMethod": "POST",
                "path": "support-chat",
                "responseMode": "responseNode"
            }
        },
        {
            "id": "agent-1",
            "name": "AI Agent",
            "type": "@n8n/n8n-nodes-langchain.agent",
            "typeVersion": 1.5,
            "position": [450, 300],
            "parameters": {
                "agent": "openAiFunctionsAgent",
                "systemMessage": (
                    "You are a customer support agent. Use the available tools "
                    "to look up order information, search the knowledge base, "
                    "and escalate issues when needed. Be concise and helpful."
                ),
                "maxIterations": 10
            }
        },
        {
            "id": "model-1",
            "name": "OpenAI GPT-4o",
            "type": "@n8n/n8n-nodes-langchain.lmChatOpenAi",
            "typeVersion": 1,
            "position": [350, 500],
            "parameters": {
                "model": "gpt-4o",
                "options": {"temperature": 0.3}
            },
            "credentials": {
                "openAiApi": {"id": "1", "name": "OpenAI account"}
            }
        },
        {
            "id": "memory-1",
            "name": "Buffer Memory",
            "type": "@n8n/n8n-nodes-langchain.memoryBufferWindow",
            "typeVersion": 1.2,
            "position": [450, 500],
            "parameters": {"sessionIdType": "fromInput", "contextWindowLength": 20}
        },
        {
            "id": "tool-rag",
            "name": "Knowledge Base Search",
            "type": "@n8n/n8n-nodes-langchain.toolVectorStore",
            "typeVersion": 1,
            "position": [550, 500],
            "parameters": {
                "name": "search_knowledge_base",
                "description": "Search the company knowledge base for product info and FAQs."
            }
        },
        {
            "id": "tool-db",
            "name": "Order Lookup",
            "type": "n8n-nodes-base.postgres",
            "typeVersion": 2.3,
            "position": [650, 500],
            "parameters": {
                "operation": "executeQuery",
                "query": "SELECT * FROM orders WHERE order_id = $1",
                "options": {"queryParameterSource": "fromAiTool"}
            }
        },
        {
            "id": "tool-slack",
            "name": "Escalate to Slack",
            "type": "n8n-nodes-base.slack",
            "typeVersion": 2.1,
            "position": [750, 500],
            "parameters": {
                "channel": "#support-escalations",
                "text": "={{ $json.escalation_reason }}"
            }
        },
        {
            "id": "respond-1",
            "name": "Respond to Webhook",
            "type": "n8n-nodes-base.respondToWebhook",
            "typeVersion": 1,
            "position": [700, 300],
            "parameters": {
                "respondWith": "json",
                "responseBody": "={{ { response: $json.output } }}"
            }
        }
    ],
    "connections": {
        "Chat Webhook": {
            "main": [[{"node": "AI Agent", "type": "main", "index": 0}]]
        },
        "AI Agent": {
            "main": [[{"node": "Respond to Webhook", "type": "main", "index": 0}]]
        },
        "OpenAI GPT-4o": {
            "ai_languageModel": [[{"node": "AI Agent", "type": "ai_languageModel", "index": 0}]]
        },
        "Buffer Memory": {
            "ai_memory": [[{"node": "AI Agent", "type": "ai_memory", "index": 0}]]
        },
        "Knowledge Base Search": {
            "ai_tool": [[{"node": "AI Agent", "type": "ai_tool", "index": 0}]]
        },
        "Order Lookup": {
            "ai_tool": [[{"node": "AI Agent", "type": "ai_tool", "index": 1}]]
        },
        "Escalate to Slack": {
            "ai_tool": [[{"node": "AI Agent", "type": "ai_tool", "index": 2}]]
        }
    },
    "settings": {"executionTimeout": 120}
}

describe_workflow(agent_workflow)

---
## Section 5: Code vs No-Code -- Same Task Both Ways

To understand the tradeoff, let us build the **same task** two ways:

> **Task**: Receive a question via HTTP, call an LLM, return a JSON response.

We will first build it in pure Python (the "code" approach),
then show the equivalent n8n workflow JSON (the "no-code" approach).

In [ ]:
# ============================================================
# APPROACH 1: Pure Python (code)
# A simple function that mirrors the webhook -> LLM -> response
# pattern. In production this would sit behind FastAPI/Flask.
# ============================================================

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def handle_chat_request_python(message: str) -> dict:
    """
    Pure Python approach: receive a message, call the LLM,
    return a structured response.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Be concise."},
            {"role": "user", "content": message},
        ],
        temperature=0.7,
        max_tokens=256,
    )
    return {
        "response": response.choices[0].message.content,
        "model": response.model,
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        },
    }


# Test the Python approach
result = handle_chat_request_python("What is n8n in one sentence?")
print("Python approach result:")
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# APPROACH 2: n8n Workflow JSON (no-code)
# The same task expressed as an n8n workflow.
# Zero Python needed -- just drag-and-drop in the n8n UI.
# ============================================================

n8n_equivalent = {
    "name": "Chat Endpoint (no-code)",
    "nodes": [
        {
            "id": "1",
            "name": "Webhook",
            "type": "n8n-nodes-base.webhook",
            "parameters": {
                "httpMethod": "POST",
                "path": "chat",
                "responseMode": "responseNode",
            },
        },
        {
            "id": "2",
            "name": "OpenAI Chat",
            "type": "@n8n/n8n-nodes-langchain.openAi",
            "parameters": {
                "model": "gpt-4o-mini",
                "prompt": "={{ $json.body.message }}",
                "systemMessage": "You are a helpful assistant. Be concise.",
                "options": {"temperature": 0.7, "maxTokens": 256},
            },
        },
        {
            "id": "3",
            "name": "Respond to Webhook",
            "type": "n8n-nodes-base.respondToWebhook",
            "parameters": {
                "respondWith": "json",
                "responseBody": "={{ { response: $json.text } }}",
            },
        },
    ],
    "connections": {
        "Webhook": {"main": [[{"node": "OpenAI Chat", "type": "main", "index": 0}]]},
        "OpenAI Chat": {"main": [[{"node": "Respond to Webhook", "type": "main", "index": 0}]]},
    },
}

print("Comparison:")
print(f"  Python approach : ~20 lines of code + FastAPI boilerplate")
print(f"  n8n approach    : 3 nodes, 0 lines of code, drag-and-drop")
print(f"  n8n JSON nodes  : {len(n8n_equivalent['nodes'])}")
print()
print("When to choose which:")
print("  - n8n : quick prototyping, non-technical team, lots of integrations")
print("  - Code: complex logic, custom algorithms, high-performance needs")
print("  - Both: n8n for orchestration, code for AI-specific complexity")

---
## Section 6: Building AI Workflows -- Webhook, LLM, Output

The most common n8n AI pattern is:

```
Trigger --> Pre-process --> LLM --> Post-process --> Output
```

Below we demonstrate this by building a **classification + response** pipeline
entirely in Python, mimicking what n8n does visually with nodes.

In [ ]:
# ============================================================
# Simulating an n8n Workflow in Python
# Pattern: Webhook -> Classify -> Route -> LLM -> Respond
# ============================================================

class WorkflowNode:
    """Base class representing a single n8n-style node."""

    def __init__(self, name: str):
        self.name = name

    def execute(self, input_data: dict) -> dict:
        raise NotImplementedError


class WebhookNode(WorkflowNode):
    """Simulates receiving a webhook payload."""

    def execute(self, input_data: dict) -> dict:
        print(f"  [{self.name}] Received: {input_data.get('message', '')[:60]}...")
        return input_data


class ClassifierNode(WorkflowNode):
    """Uses an LLM to classify the incoming message."""

    CATEGORIES = ["billing", "technical", "general", "complaint"]

    def execute(self, input_data: dict) -> dict:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        f"Classify this customer message into exactly one category: "
                        f"{', '.join(self.CATEGORIES)}. Respond with just the category name."
                    ),
                },
                {"role": "user", "content": input_data["message"]},
            ],
            max_tokens=20,
            temperature=0,
        )
        category = resp.choices[0].message.content.strip().lower()
        input_data["category"] = category
        print(f"  [{self.name}] Category: {category}")
        return input_data


class LLMResponseNode(WorkflowNode):
    """Generates a response based on the category."""

    SYSTEM_PROMPTS = {
        "billing": "You are a billing specialist. Help with invoices, payments, and refunds.",
        "technical": "You are a technical support expert. Help with product issues.",
        "general": "You are a friendly customer service rep. Answer general questions.",
        "complaint": "You are a senior customer advocate. Acknowledge the issue and offer resolution.",
    }

    def execute(self, input_data: dict) -> dict:
        category = input_data.get("category", "general")
        system_prompt = self.SYSTEM_PROMPTS.get(category, self.SYSTEM_PROMPTS["general"])
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": input_data["message"]},
            ],
            max_tokens=256,
            temperature=0.7,
        )
        input_data["response"] = resp.choices[0].message.content
        print(f"  [{self.name}] Generated response ({len(input_data['response'])} chars)")
        return input_data


class RespondNode(WorkflowNode):
    """Formats the final output."""

    def execute(self, input_data: dict) -> dict:
        output = {
            "category": input_data.get("category", "unknown"),
            "response": input_data.get("response", ""),
        }
        print(f"  [{self.name}] Sending response")
        return output


# Build the pipeline (mirrors the n8n canvas)
pipeline = [
    WebhookNode("Webhook"),
    ClassifierNode("Classifier"),
    LLMResponseNode("LLM Response"),
    RespondNode("Respond"),
]


def run_workflow(payload: dict) -> dict:
    """Execute the workflow: data flows through each node sequentially."""
    data = payload
    for node in pipeline:
        data = node.execute(data)
    return data


# Test with sample messages
test_messages = [
    "I was charged twice for my subscription last month.",
    "How do I reset my password?",
    "Your product is terrible and I want a refund immediately!",
]

for msg in test_messages:
    print(f"\nInput: {msg}")
    result = run_workflow({"message": msg})
    print(f"Output: [{result['category']}] {result['response'][:120]}...")
    print("-" * 70)

---
## Section 7: RAG Workflows in n8n

A complete RAG pipeline in n8n requires **two workflows**:

**Ingestion workflow:**
```
Google Drive Trigger -> Download -> Document Loader -> Text Splitter -> Embeddings -> Vector Store Insert -> Slack Notify
```

**Query workflow:**
```
Webhook (POST /query) -> Vector Store Retriever (top 5) -> OpenAI Chat (answer with context) -> Respond
```

Below we define both as n8n-compatible JSON and show the Python equivalent of the query workflow.

In [ ]:
# ============================================================
# n8n RAG Ingestion Workflow JSON
# ============================================================

rag_ingestion_workflow = {
    "name": "RAG Document Ingestion",
    "nodes": [
        {
            "id": "1",
            "name": "Google Drive Trigger",
            "type": "n8n-nodes-base.googleDriveTrigger",
            "parameters": {
                "event": "fileCreated",
                "folderId": "your-folder-id",
                "pollTimes": {"item": [{"mode": "everyMinute"}]}
            }
        },
        {
            "id": "2",
            "name": "Download File",
            "type": "n8n-nodes-base.googleDrive",
            "parameters": {"operation": "download", "fileId": "={{ $json.id }}"}
        },
        {
            "id": "3",
            "name": "Extract Text",
            "type": "@n8n/n8n-nodes-langchain.documentDefaultDataLoader",
            "parameters": {"dataType": "binary"}
        },
        {
            "id": "4",
            "name": "Text Splitter",
            "type": "@n8n/n8n-nodes-langchain.textSplitterRecursiveCharacterTextSplitter",
            "parameters": {"chunkSize": 1000, "chunkOverlap": 200}
        },
        {
            "id": "5",
            "name": "OpenAI Embeddings",
            "type": "@n8n/n8n-nodes-langchain.embeddingsOpenAi",
            "parameters": {"model": "text-embedding-3-small"}
        },
        {
            "id": "6",
            "name": "Pinecone Insert",
            "type": "@n8n/n8n-nodes-langchain.vectorStorePinecone",
            "parameters": {"mode": "insert", "pineconeIndex": "docs-index"}
        },
        {
            "id": "7",
            "name": "Slack Notify",
            "type": "n8n-nodes-base.slack",
            "parameters": {
                "channel": "#knowledge-base",
                "text": "New document indexed: {{ $json.fileName }}"
            }
        }
    ],
    "connections": {
        "Google Drive Trigger": {"main": [[{"node": "Download File", "type": "main", "index": 0}]]},
        "Download File": {"main": [[{"node": "Extract Text", "type": "main", "index": 0}]]},
        "Extract Text": {"main": [[{"node": "Pinecone Insert", "type": "main", "index": 0}]]},
        "Text Splitter": {"ai_textSplitter": [[{"node": "Extract Text", "type": "ai_textSplitter", "index": 0}]]},
        "OpenAI Embeddings": {"ai_embedding": [[{"node": "Pinecone Insert", "type": "ai_embedding", "index": 0}]]},
        "Pinecone Insert": {"main": [[{"node": "Slack Notify", "type": "main", "index": 0}]]}
    }
}

describe_workflow(rag_ingestion_workflow)

In [ ]:
# ============================================================
# Python equivalent of the n8n RAG Query workflow
# Simulates: Webhook -> Retrieve context -> LLM answer -> Respond
# ============================================================

# Simulated knowledge base (in production: Pinecone, Qdrant, etc.)
MOCK_KNOWLEDGE_BASE = [
    {
        "text": "Our return policy allows returns within 30 days of purchase. "
                "Items must be in original packaging with receipt.",
        "source": "returns-policy.pdf",
    },
    {
        "text": "Enterprise plans include SSO, audit logs, dedicated support, "
                "and custom SLAs. Pricing starts at $500/month.",
        "source": "pricing-guide.pdf",
    },
    {
        "text": "To reset your password, go to Settings > Security > Change Password. "
                "If locked out, use the 'Forgot Password' link on the login page.",
        "source": "user-guide.pdf",
    },
    {
        "text": "API rate limits: Free tier is 100 requests/minute. Pro tier is "
                "1000 requests/minute. Enterprise has no hard limit.",
        "source": "api-docs.pdf",
    },
]


def simple_retriever(query: str, top_k: int = 2) -> list[dict]:
    """
    Naive keyword retriever (in production you would use embeddings + vector store).
    Mirrors what the n8n Vector Store Retriever node does.
    """
    scored = []
    query_words = set(query.lower().split())
    for doc in MOCK_KNOWLEDGE_BASE:
        doc_words = set(doc["text"].lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


def rag_query(question: str) -> dict:
    """
    RAG pipeline: retrieve context, then generate an answer.
    This mirrors the n8n query workflow:
      Webhook -> Vector Store Retriever -> OpenAI Chat -> Respond
    """
    # Step 1: Retrieve (n8n: Vector Store Retriever node)
    retrieved = simple_retriever(question, top_k=2)
    context = "\n\n".join(
        f"[Source: {doc['source']}] {doc['text']}" for doc in retrieved
    )

    # Step 2: Generate (n8n: OpenAI Chat node)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the user's question based on the provided context. "
                    "If the context does not contain relevant information, say so. "
                    "Cite the source document."
                ),
            },
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        max_tokens=256,
        temperature=0.3,
    )

    return {
        "answer": resp.choices[0].message.content,
        "sources": [doc["source"] for doc in retrieved],
    }


# Test the RAG pipeline
questions = [
    "What is your return policy?",
    "How do I reset my password?",
    "What are the API rate limits for the Pro tier?",
]

for q in questions:
    print(f"Q: {q}")
    result = rag_query(q)
    print(f"A: {result['answer']}")
    print(f"   Sources: {result['sources']}")
    print("-" * 70)

---
## Section 8: n8n Credential Management

n8n encrypts all credentials at rest using `N8N_ENCRYPTION_KEY`. Key points:

- **Never store API keys in workflow JSON** -- use n8n's credential system
- When exporting workflows, credentials are excluded by default
- Set the encryption key once and never change it (invalidates all stored credentials)
- For CI/CD, inject credentials via the n8n API after deployment

Below we show how to manage credentials programmatically.

In [ ]:
# ============================================================
# Credential management via n8n API
# ============================================================

def create_openai_credential(n8n_client: N8NClient, api_key: str, name: str = "OpenAI") -> dict:
    """
    Create an OpenAI credential in n8n via the API.
    The API key is encrypted at rest by n8n.
    """
    credential_payload = {
        "name": name,
        "type": "openAiApi",
        "data": {
            "apiKey": api_key,
        },
    }
    resp = requests.post(
        f"{n8n_client.base_url}/api/v1/credentials",
        headers=n8n_client.headers,
        json=credential_payload,
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


def list_credentials(n8n_client: N8NClient) -> list[dict]:
    """List all stored credentials (keys are never returned)."""
    resp = requests.get(
        f"{n8n_client.base_url}/api/v1/credentials",
        headers=n8n_client.headers,
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json().get("data", [])


# Security best practices
print("n8n Credential Security Best Practices:")
print("="*50)
print("1. Set N8N_ENCRYPTION_KEY as an environment variable")
print("2. Never export workflows with --include-credentials")
print("3. Use the n8n API to inject credentials in CI/CD")
print("4. Rotate credentials via the n8n UI or API periodically")
print("5. Use separate n8n credentials per environment (dev/staging/prod)")
print("6. The API never returns credential secrets -- only metadata")

---
## Section 9: Error Handling in Workflows

n8n supports robust error handling:
- **Error Workflow**: a special workflow that triggers when any other workflow fails
- **Retry on Fail**: per-node retry with configurable attempts and backoff
- **Error Output**: route errors to a different branch (try/catch pattern)

Below we demonstrate a retry-with-backoff pattern and error routing in Python,
mirroring what n8n provides visually.

In [ ]:
# ============================================================
# Error handling patterns for n8n-style workflows
# ============================================================

import random


def retry_with_backoff(
    func,
    *args,
    max_retries: int = 3,
    initial_delay: float = 1.0,
    backoff_factor: float = 2.0,
    **kwargs,
) -> Any:
    """
    Retry a function with exponential backoff.
    Mirrors n8n's per-node 'Retry On Fail' setting.
    """
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                delay = initial_delay * (backoff_factor ** attempt)
                print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"  All {max_retries + 1} attempts failed.")
    raise last_error


def unreliable_api_call(fail_rate: float = 0.7) -> dict:
    """Simulates a flaky API that fails 70% of the time."""
    if random.random() < fail_rate:
        raise ConnectionError("API timeout")
    return {"status": "ok", "data": "success"}


# Demo: retry with backoff
print("Retry with backoff demo:")
try:
    result = retry_with_backoff(unreliable_api_call, fail_rate=0.5, max_retries=3, initial_delay=0.1)
    print(f"  Success: {result}")
except ConnectionError:
    print("  Final failure after all retries.")

In [ ]:
# ============================================================
# Error Workflow Pattern
# In n8n, you designate a workflow as the "Error Workflow".
# It receives the failed execution context and can send alerts.
# ============================================================

error_workflow_json = {
    "name": "Global Error Handler",
    "nodes": [
        {
            "id": "1",
            "name": "Error Trigger",
            "type": "n8n-nodes-base.errorTrigger",
            "parameters": {},
        },
        {
            "id": "2",
            "name": "Format Error Message",
            "type": "n8n-nodes-base.set",
            "parameters": {
                "values": {
                    "string": [
                        {
                            "name": "alert_text",
                            "value": "Workflow '{{ $json.workflow.name }}' failed. "
                                     "Error: {{ $json.execution.error.message }}",
                        }
                    ]
                }
            },
        },
        {
            "id": "3",
            "name": "Send Slack Alert",
            "type": "n8n-nodes-base.slack",
            "parameters": {
                "channel": "#alerts",
                "text": "={{ $json.alert_text }}",
            },
        },
        {
            "id": "4",
            "name": "Send Email Alert",
            "type": "n8n-nodes-base.emailSend",
            "parameters": {
                "fromEmail": "alerts@company.com",
                "toEmail": "oncall@company.com",
                "subject": "n8n Workflow Failure Alert",
                "text": "={{ $json.alert_text }}",
            },
        },
    ],
    "connections": {
        "Error Trigger": {"main": [[{"node": "Format Error Message", "type": "main", "index": 0}]]},
        "Format Error Message": {
            "main": [
                [
                    {"node": "Send Slack Alert", "type": "main", "index": 0},
                    {"node": "Send Email Alert", "type": "main", "index": 0},
                ]
            ]
        },
    },
}

print("Error Workflow structure:")
describe_workflow(error_workflow_json)
print("\nThis workflow triggers automatically when any other workflow fails.")
print("Set it in n8n: Settings > Error Workflow.")

---
## Section 10: Common Workflow Patterns

Three high-value patterns for GenAI teams:

1. **Document Processing Pipeline** -- ingest, summarize, and store documents
2. **Chatbot with Tool Use** -- agent-based customer support
3. **Data Enrichment Pipeline** -- enrich CRM records with AI research

In [ ]:
# ============================================================
# Pattern 1: Document Processing Pipeline
# Simulates: Upload -> Extract -> Summarize -> Store -> Notify
# ============================================================

def document_processing_pipeline(document_text: str, filename: str) -> dict:
    """
    Simulates the n8n pattern:
      File Upload -> Document Loader -> LLM Summarize -> Database Insert -> Slack
    """
    print(f"Processing: {filename}")

    # Step 1: Chunk the document (n8n: Text Splitter node)
    chunk_size = 500
    chunks = [
        document_text[i : i + chunk_size]
        for i in range(0, len(document_text), chunk_size)
    ]
    print(f"  Chunked into {len(chunks)} pieces")

    # Step 2: Summarize (n8n: OpenAI Chat node)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Summarize the document in 2-3 sentences. Focus on key points.",
            },
            {"role": "user", "content": document_text[:2000]},
        ],
        max_tokens=150,
        temperature=0.3,
    )
    summary = resp.choices[0].message.content
    print(f"  Summary generated")

    # Step 3: Store metadata (n8n: PostgreSQL Insert node)
    record = {
        "filename": filename,
        "chunk_count": len(chunks),
        "summary": summary,
        "status": "indexed",
    }
    print(f"  Record stored")

    # Step 4: Notify (n8n: Slack node)
    notification = f"New document indexed: {filename} ({len(chunks)} chunks)"
    print(f"  Notification: {notification}")

    return record


# Test
sample_doc = (
    "n8n is an open-source workflow automation platform that provides a visual "
    "interface for building complex automation workflows. It supports over 400 "
    "integrations including Slack, Gmail, databases, and AI services. The platform "
    "can be self-hosted for data privacy and compliance requirements. It includes "
    "native AI nodes for building LLM-powered applications including RAG pipelines "
    "and autonomous agents. n8n workflows are stored as JSON and can be version "
    "controlled with Git."
)

result = document_processing_pipeline(sample_doc, "n8n-overview.pdf")
print(f"\nResult: {json.dumps(result, indent=2)}")

In [ ]:
# ============================================================
# Pattern 2: Data Enrichment Pipeline
# New CRM lead -> AI research -> Update CRM record
# ============================================================

def data_enrichment_pipeline(lead: dict) -> dict:
    """
    Simulates the n8n pattern:
      CRM Trigger (new lead) -> AI Research -> CRM Update
    """
    print(f"Enriching lead: {lead['company']}")

    # Step 1: AI Research (n8n: OpenAI Chat node)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sales research assistant. Given a company name, "
                    "provide a brief summary of what the company does, its industry, "
                    "approximate size, and why they might need AI automation. "
                    "Be concise (3-4 sentences). Use your general knowledge."
                ),
            },
            {"role": "user", "content": f"Research this company: {lead['company']}"},
        ],
        max_tokens=200,
        temperature=0.5,
    )
    research = resp.choices[0].message.content

    # Step 2: Score the lead (n8n: Code node with custom logic)
    score_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Rate this lead 1-10 for AI automation potential. Respond with just the number.",
            },
            {"role": "user", "content": research},
        ],
        max_tokens=5,
        temperature=0,
    )
    score = score_resp.choices[0].message.content.strip()

    enriched = {
        **lead,
        "research_summary": research,
        "ai_score": score,
        "status": "enriched",
    }
    print(f"  Score: {score}/10")
    print(f"  Research: {research[:120]}...")
    return enriched


# Test
test_leads = [
    {"company": "Stripe", "contact": "Jane Doe", "email": "jane@stripe.com"},
    {"company": "Mayo Clinic", "contact": "John Smith", "email": "john@mayo.edu"},
]

for lead in test_leads:
    enriched = data_enrichment_pipeline(lead)
    print("-" * 70)

In [ ]:
# ============================================================
# Pattern 3: Email Triage with AI Classification
# Gmail Trigger -> Classify -> Route -> Auto-respond or Escalate
# ============================================================

def email_triage(email: dict) -> dict:
    """
    Simulates n8n email triage pattern:
      Gmail Trigger -> AI Classify -> IF urgent -> Slack escalation
                                   -> ELSE -> Draft auto-reply
    """
    # Step 1: Classify (n8n: OpenAI Chat node)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify this email. Respond with JSON: "
                    '{"urgency": "low|medium|high", "category": "support|billing|sales|spam", '
                    '"auto_reply_ok": true/false}'
                ),
            },
            {
                "role": "user",
                "content": f"Subject: {email['subject']}\nBody: {email['body']}",
            },
        ],
        max_tokens=100,
        temperature=0,
    )

    try:
        classification = json.loads(resp.choices[0].message.content)
    except json.JSONDecodeError:
        classification = {"urgency": "medium", "category": "support", "auto_reply_ok": False}

    # Step 2: Route (n8n: IF node)
    if classification.get("urgency") == "high":
        action = "escalate_to_slack"
        message = f"URGENT email from {email['from']}: {email['subject']}"
    elif classification.get("auto_reply_ok"):
        action = "auto_reply"
        # Generate auto-reply (n8n: OpenAI Chat node on the auto-reply branch)
        reply_resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Draft a brief, professional reply to this email."},
                {"role": "user", "content": email["body"]},
            ],
            max_tokens=150,
        )
        message = reply_resp.choices[0].message.content
    else:
        action = "queue_for_review"
        message = "Queued for human review"

    return {
        "from": email["from"],
        "subject": email["subject"],
        "classification": classification,
        "action": action,
        "message": message[:200],
    }


# Test
test_emails = [
    {
        "from": "customer@example.com",
        "subject": "URGENT: Production system down",
        "body": "Our entire production system is down since this morning. We need immediate help.",
    },
    {
        "from": "user@example.com",
        "subject": "How to export data?",
        "body": "Hi, can you tell me how to export my data to CSV format? Thanks.",
    },
]

for email in test_emails:
    result = email_triage(email)
    print(f"From: {result['from']}")
    print(f"Classification: {result['classification']}")
    print(f"Action: {result['action']}")
    print(f"Message: {result['message'][:120]}...")
    print("-" * 70)

---
## Section 11: Integration Examples

n8n has 400+ integrations. Below are workflow JSONs for the most common
AI-powered integration patterns: Slack, email, and databases.

In [ ]:
# ============================================================
# Integration Pattern: Slack Bot with AI
# Slack slash command -> AI Agent -> Slack reply
# ============================================================

slack_bot_workflow = {
    "name": "Slack AI Assistant",
    "nodes": [
        {
            "id": "1",
            "name": "Slack Trigger",
            "type": "n8n-nodes-base.slackTrigger",
            "parameters": {
                "event": "message",
                "channelId": "C_YOUR_CHANNEL",
            },
        },
        {
            "id": "2",
            "name": "Filter Bot Messages",
            "type": "n8n-nodes-base.if",
            "parameters": {
                "conditions": {
                    "string": [{"value1": "={{ $json.bot_id }}", "operation": "isEmpty"}]
                }
            },
        },
        {
            "id": "3",
            "name": "AI Response",
            "type": "@n8n/n8n-nodes-langchain.openAi",
            "parameters": {
                "model": "gpt-4o-mini",
                "prompt": "={{ $json.text }}",
                "systemMessage": "You are a helpful Slack assistant. Be concise.",
            },
        },
        {
            "id": "4",
            "name": "Post Reply",
            "type": "n8n-nodes-base.slack",
            "parameters": {
                "channel": "={{ $json.channel }}",
                "text": "={{ $('AI Response').item.json.text }}",
                "options": {"thread_ts": "={{ $json.ts }}"},
            },
        },
    ],
    "connections": {
        "Slack Trigger": {"main": [[{"node": "Filter Bot Messages", "type": "main", "index": 0}]]},
        "Filter Bot Messages": {"main": [[{"node": "AI Response", "type": "main", "index": 0}]]},
        "AI Response": {"main": [[{"node": "Post Reply", "type": "main", "index": 0}]]},
    },
}

# Database monitoring workflow
db_monitor_workflow = {
    "name": "Database Anomaly Monitor",
    "nodes": [
        {
            "id": "1",
            "name": "Schedule (every 5 min)",
            "type": "n8n-nodes-base.scheduleTrigger",
            "parameters": {"rule": {"interval": [{"field": "minutes", "minutesInterval": 5}]}},
        },
        {
            "id": "2",
            "name": "Query Metrics",
            "type": "n8n-nodes-base.postgres",
            "parameters": {
                "operation": "executeQuery",
                "query": (
                    "SELECT metric_name, value, recorded_at "
                    "FROM metrics WHERE recorded_at > NOW() - INTERVAL '5 minutes'"
                ),
            },
        },
        {
            "id": "3",
            "name": "AI Anomaly Analysis",
            "type": "@n8n/n8n-nodes-langchain.openAi",
            "parameters": {
                "model": "gpt-4o-mini",
                "prompt": "Analyze these metrics for anomalies: {{ JSON.stringify($json) }}",
                "systemMessage": (
                    "You are a monitoring analyst. Identify any anomalies in the metrics. "
                    "Respond with JSON: {\"anomaly\": true/false, \"details\": \"...\"}"
                ),
            },
        },
        {
            "id": "4",
            "name": "Check Anomaly",
            "type": "n8n-nodes-base.if",
            "parameters": {
                "conditions": {
                    "string": [{"value1": "={{ $json.text }}", "value2": "true", "operation": "contains"}]
                }
            },
        },
        {
            "id": "5",
            "name": "PagerDuty Alert",
            "type": "n8n-nodes-base.httpRequest",
            "parameters": {
                "method": "POST",
                "url": "https://events.pagerduty.com/v2/enqueue",
                "body": "{\"routing_key\": \"...\", \"event_action\": \"trigger\"}",
            },
        },
    ],
    "connections": {
        "Schedule (every 5 min)": {"main": [[{"node": "Query Metrics", "type": "main", "index": 0}]]},
        "Query Metrics": {"main": [[{"node": "AI Anomaly Analysis", "type": "main", "index": 0}]]},
        "AI Anomaly Analysis": {"main": [[{"node": "Check Anomaly", "type": "main", "index": 0}]]},
        "Check Anomaly": {"main": [[{"node": "PagerDuty Alert", "type": "main", "index": 0}]]},
    },
}

print("=" * 50)
describe_workflow(slack_bot_workflow)
print("\n" + "=" * 50)
describe_workflow(db_monitor_workflow)

---
## Section 12: When to Use No-Code vs Code

This is the key decision framework for GenAI teams.

| Criterion | Use n8n (No-Code) | Use Python (Code) |
|---|---|---|
| **Who builds it** | Business users, product managers | Engineers, data scientists |
| **Iteration speed** | Minutes to change a workflow | Hours to change, test, deploy |
| **Integration count** | 5+ external services | 0-2 external services |
| **AI complexity** | Standard LLM calls, basic RAG | Custom agents, fine-tuned models |
| **Data volume** | Low-medium (< 10K items/run) | High (millions of records) |
| **Latency needs** | Seconds OK (async workflows) | Milliseconds needed |
| **Compliance** | Audit trail built-in | Custom logging required |
| **Version control** | JSON export + Git | Native Git workflow |

**The best teams use both**: n8n for orchestration and integration plumbing,
Python for the core AI logic. Connect them via webhooks or the n8n Code node.

In [ ]:
# ============================================================
# Decision helper: should this task be n8n or code?
# ============================================================

def recommend_approach(task_description: str) -> dict:
    """
    Uses an LLM to recommend whether a task should be built
    with n8n (no-code), Python (code), or a hybrid approach.
    """
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": textwrap.dedent("""\
                    You are a solutions architect choosing between n8n (no-code workflow
                    automation) and Python code for AI tasks. Analyze the task and respond
                    with JSON:
                    {
                      "recommendation": "n8n" | "code" | "hybrid",
                      "confidence": 0.0-1.0,
                      "reasoning": "one sentence",
                      "n8n_nodes": ["list of n8n nodes you would use"],
                      "python_libs": ["list of Python libraries if code is needed"]
                    }
                """),
            },
            {"role": "user", "content": task_description},
        ],
        max_tokens=250,
        temperature=0.3,
    )

    try:
        return json.loads(resp.choices[0].message.content)
    except json.JSONDecodeError:
        return {"raw": resp.choices[0].message.content}


# Test with different task types
tasks = [
    "When a customer emails support, classify the email, search our knowledge base, and reply automatically.",
    "Fine-tune a LoRA adapter on domain-specific medical texts and evaluate with ROUGE metrics.",
    "Watch a Google Drive folder for new PDFs, extract text, generate embeddings, store in Pinecone, and notify Slack.",
    "Build a multi-agent system with plan-and-execute architecture for complex research tasks.",
]

for task in tasks:
    print(f"Task: {task[:80]}...")
    rec = recommend_approach(task)
    print(f"  Recommendation: {rec.get('recommendation', 'N/A')}")
    print(f"  Confidence: {rec.get('confidence', 'N/A')}")
    print(f"  Reasoning: {rec.get('reasoning', 'N/A')}")
    if rec.get("n8n_nodes"):
        print(f"  n8n nodes: {', '.join(rec['n8n_nodes'])}")
    if rec.get("python_libs"):
        print(f"  Python libs: {', '.join(rec['python_libs'])}")
    print("-" * 70)

---
## Section 13: Triggers & Integration Reference

Quick reference for the most useful n8n triggers in AI workflows:

| Trigger Type | n8n Nodes | AI Use Case |
|---|---|---|
| Webhook | Webhook, Chat Trigger | Chatbot API, form submission AI |
| Schedule | Schedule Trigger, Cron | Daily report generation, batch processing |
| Email | Gmail, Outlook Trigger | Email classification, auto-response |
| Messaging | Slack, Teams, Discord | Slash command bots, channel assistants |
| File | Google Drive, S3, FTP | Document ingestion, media processing |
| Database | PostgreSQL, MySQL Trigger | New record enrichment, change analysis |
| CRM | HubSpot, Salesforce | Lead scoring, account research |

In [ ]:
# ============================================================
# Trigger configuration examples as n8n JSON
# ============================================================

trigger_examples = {
    "webhook_chat": {
        "type": "n8n-nodes-base.webhook",
        "parameters": {
            "httpMethod": "POST",
            "path": "chat",
            "responseMode": "responseNode",
            "options": {"rawBody": True},
        },
        "use_case": "Chatbot endpoint -- receives user messages via HTTP POST",
    },
    "schedule_daily": {
        "type": "n8n-nodes-base.scheduleTrigger",
        "parameters": {
            "rule": {"interval": [{"field": "hours", "hoursInterval": 24, "triggerAtHour": 8}]}
        },
        "use_case": "Daily 8 AM report generation",
    },
    "gmail_new_email": {
        "type": "n8n-nodes-base.gmailTrigger",
        "parameters": {
            "pollTimes": {"item": [{"mode": "everyMinute"}]},
            "filters": {"labelIds": ["INBOX"]},
        },
        "use_case": "Email triage -- classify and route incoming emails",
    },
    "slack_message": {
        "type": "n8n-nodes-base.slackTrigger",
        "parameters": {
            "event": "message",
            "channelId": "C_CHANNEL_ID",
        },
        "use_case": "Slack AI bot -- responds to messages in a channel",
    },
    "google_drive_file": {
        "type": "n8n-nodes-base.googleDriveTrigger",
        "parameters": {
            "event": "fileCreated",
            "folderId": "FOLDER_ID",
        },
        "use_case": "Document ingestion -- process new files uploaded to Drive",
    },
}

print("n8n Trigger Reference:")
print("=" * 60)
for name, config in trigger_examples.items():
    print(f"\n  {name}")
    print(f"    Type:     {config['type']}")
    print(f"    Use case: {config['use_case']}")

---
## Summary

In this notebook we covered:

1. **n8n installation** -- Docker and npm setup, plus production Docker Compose with PostgreSQL and Redis
2. **Workflow JSON structure** -- nodes, connections, and settings that define an n8n workflow
3. **n8n REST API** -- programmatic workflow creation, activation, and triggering
4. **AI & LLM nodes** -- the full AI node catalog and how to wire an Agent with tools and memory
5. **Code vs no-code comparison** -- same webhook-to-LLM task built both ways
6. **Workflow simulation** -- Python classes that mirror n8n's node-based execution model
7. **RAG workflows** -- ingestion and query pipelines as n8n JSON and Python equivalents
8. **Credential management** -- encryption, API-based credential injection, security best practices
9. **Error handling** -- retry with backoff, error workflows, Slack/email alerting
10. **Common patterns** -- document processing, data enrichment, email triage
11. **Integration examples** -- Slack bot, database anomaly monitor with PagerDuty
12. **Decision framework** -- when to use n8n, when to use code, when to use both

**Key takeaway**: n8n handles orchestration and integration plumbing; Python handles core AI complexity. The best GenAI teams use both.

**Next module**: `15-capstone-document-portal.html` -- Capstone I: Document Portal